In [1]:
# ==================== KHỞI TẠO SPARK ====================
import os
# Không cần set JAVA_HOME nếu đã cài Java 17 đúng
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, avg, count, max as spark_max
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

spark = SparkSession.builder.appName("MovieRatingAnalysis").getOrCreate()

# Đọc file movies.txt
movies_schema = StructType([
    StructField("MovieID", IntegerType(), True),
    StructField("Title", StringType(), True),
    StructField("Genres", StringType(), True)
])
movies_df = spark.read.option("delimiter", ", ").schema(movies_schema).csv("movies.txt")

# Đọc users.txt
users_schema = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Occupation", IntegerType(), True),
    StructField("Zip", StringType(), True)
])
users_df = spark.read.option("delimiter", ", ").schema(users_schema).csv("users.txt")

# Đọc ratings từ 2 file
ratings_schema = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("MovieID", IntegerType(), True),
    StructField("Rating", FloatType(), True),
    StructField("Timestamp", IntegerType(), True)
])
ratings1_df = spark.read.option("delimiter", ", ").schema(ratings_schema).csv("ratings_1.txt")
ratings2_df = spark.read.option("delimiter", ", ").schema(ratings_schema).csv("ratings_2.txt")
ratings_df = ratings1_df.union(ratings2_df).select("UserID", "MovieID", "Rating")

# Đăng ký temp view
movies_df.createOrReplaceTempView("movies")
users_df.createOrReplaceTempView("users")
ratings_df.createOrReplaceTempView("ratings")

print("Spark ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/22 00:01:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready


## Bài 01

In [2]:
# Tính avg_rating và total_ratings cho mỗi phim
movie_stats = ratings_df.groupBy("MovieID").agg(
    avg("Rating").alias("avg_rating"),
    count("Rating").alias("total_ratings")
)

# Kết quả chi tiết
result1 = movie_stats.join(movies_df, "MovieID").select("Title", "avg_rating", "total_ratings").orderBy("Title")
print("Bài 1 - Kết quả:")
for row in result1.collect():
    print(f"{row.Title} AverageRating: {row.avg_rating:.2f} (TotalRatings: {row.total_ratings})")

# Phim có avg_rating cao nhất (>=5 ratings)
from pyspark.sql.functions import max as spark_max
max_avg = movie_stats.filter(col("total_ratings") >= 5).agg(spark_max("avg_rating").alias("max_avg")).collect()[0][0]
best_movie = movie_stats.filter((col("total_ratings") >= 5) & (col("avg_rating") == max_avg)).join(movies_df, "MovieID").select("Title", "avg_rating").first()
if best_movie:
    print(f"\n{best_movie.Title} is the highest rated movie with an average rating of {best_movie.avg_rating:.2f} among movies with at least 5 ratings.")

Bài 1 - Kết quả:
American Beauty (1999) AverageRating: 3.75 (TotalRatings: 2)
Avatar (2009) AverageRating: 4.75 (TotalRatings: 2)
Back to the Future (1985) AverageRating: 4.00 (TotalRatings: 2)
Birdman (2014) AverageRating: 3.00 (TotalRatings: 2)
Braveheart (1995) AverageRating: 4.25 (TotalRatings: 2)
Coco (2017) AverageRating: 4.25 (TotalRatings: 2)
Dunkirk (2017) AverageRating: 3.50 (TotalRatings: 2)
Fight Club (1999) AverageRating: 4.25 (TotalRatings: 2)
Forrest Gump (1994) AverageRating: 3.75 (TotalRatings: 2)
Gladiator (2000) AverageRating: 3.25 (TotalRatings: 2)
Good Will Hunting (1997) AverageRating: 3.50 (TotalRatings: 2)
Gravity (2013) AverageRating: 4.25 (TotalRatings: 2)
Inception (2010) AverageRating: 3.75 (TotalRatings: 2)
Interstellar (2014) AverageRating: 3.50 (TotalRatings: 2)
Jurassic Park (1993) AverageRating: 4.00 (TotalRatings: 2)
La La Land (2016) AverageRating: 4.00 (TotalRatings: 2)
Mad Max: Fury Road (2015) AverageRating: 4.00 (TotalRatings: 2)
Manchester by the

## Bài 02

In [3]:
# Ghép ratings với movies để lấy genres
ratings_with_genres = ratings_df.join(movies_df, "MovieID").select("Rating", "Genres")
# Tách genres
genres_exploded = ratings_with_genres.withColumn("Genre", explode(split(col("Genres"), "\\|"))).select("Genre", "Rating")
genre_stats = genres_exploded.groupBy("Genre").agg(avg("Rating").alias("avg_rating"), count("Rating").alias("total_ratings")).orderBy("Genre")
print("\nBài 2 - Kết quả:")
for row in genre_stats.collect():
    print(f"{row.Genre}: {row.avg_rating:.2f} (TotalRatings: {row.total_ratings})")


Bài 2 - Kết quả:
Action: 3.73 (TotalRatings: 20)
Adventure: 3.85 (TotalRatings: 30)
Animation: 4.17 (TotalRatings: 6)
Biography: 4.14 (TotalRatings: 14)
Children: 4.25 (TotalRatings: 2)
Comedy: 3.96 (TotalRatings: 12)
Crime: 3.77 (TotalRatings: 20)
Drama: 3.81 (TotalRatings: 76)
Family: 4.25 (TotalRatings: 2)
Fantasy: 3.95 (TotalRatings: 10)
History: 4.00 (TotalRatings: 6)
Music: 3.88 (TotalRatings: 4)
Mystery: 4.00 (TotalRatings: 8)
Romance: 3.75 (TotalRatings: 8)
Sci-Fi: 3.85 (TotalRatings: 20)
Thriller: 3.88 (TotalRatings: 16)
War: 4.00 (TotalRatings: 2)


## Bài 03

In [4]:
# Join ratings với users
ratings_users = ratings_df.join(users_df, "UserID").select("MovieID", "Rating", "Gender")
# Tính avg theo giới
male_avg = ratings_users.filter(col("Gender") == "M").groupBy("MovieID").agg(avg("Rating").alias("male_avg"))
female_avg = ratings_users.filter(col("Gender") == "F").groupBy("MovieID").agg(avg("Rating").alias("female_avg"))
gender_stats = male_avg.join(female_avg, "MovieID", "full").fillna(0).join(movies_df, "MovieID").select("Title", "male_avg", "female_avg").orderBy("Title")
print("\nBài 3 - Kết quả:")
for row in gender_stats.collect():
    male = f"{row.male_avg:.2f}" if row.male_avg != 0 else "N/A"
    female = f"{row.female_avg:.2f}" if row.female_avg != 0 else "N/A"
    print(f"{row.Title}: Male_Avg = {male}, Female_Avg = {female}")


Bài 3 - Kết quả:
American Beauty (1999): Male_Avg = 3.00, Female_Avg = 4.50
Avatar (2009): Male_Avg = 5.00, Female_Avg = 4.50
Back to the Future (1985): Male_Avg = 4.00, Female_Avg = 4.00
Birdman (2014): Male_Avg = 3.00, Female_Avg = 3.00
Braveheart (1995): Male_Avg = 4.50, Female_Avg = 4.00
Coco (2017): Male_Avg = 5.00, Female_Avg = 3.50
Dunkirk (2017): Male_Avg = 4.00, Female_Avg = 3.00
Fight Club (1999): Male_Avg = 5.00, Female_Avg = 3.50
Forrest Gump (1994): Male_Avg = 2.50, Female_Avg = 5.00
Gladiator (2000): Male_Avg = 3.50, Female_Avg = 3.00
Good Will Hunting (1997): Male_Avg = 3.00, Female_Avg = 4.00
Gravity (2013): Male_Avg = 4.00, Female_Avg = 4.50
Inception (2010): Male_Avg = 3.50, Female_Avg = 4.00
Interstellar (2014): Male_Avg = 3.50, Female_Avg = 3.50
Jurassic Park (1993): Male_Avg = 3.50, Female_Avg = 4.50
La La Land (2016): Male_Avg = 5.00, Female_Avg = 3.00
Mad Max: Fury Road (2015): Male_Avg = 4.00, Female_Avg = 4.00
Manchester by the Sea (2016): Male_Avg = 3.50, Fem

## Bài 04

In [5]:
from pyspark.sql.functions import udf
def age_group(age):
    if age < 18: return '0-18'
    elif age < 35: return '18-35'
    elif age < 50: return '35-50'
    else: return '50+'
age_udf = udf(age_group, StringType())

ratings_users_age = ratings_df.join(users_df, "UserID").withColumn("AgeGroup", age_udf(col("Age"))).select("MovieID", "Rating", "AgeGroup")
age_stats = ratings_users_age.groupBy("MovieID", "AgeGroup").agg(avg("Rating").alias("avg_rating"))
pivot_stats = age_stats.groupBy("MovieID").pivot("AgeGroup", ["0-18", "18-35", "35-50", "50+"]).agg(avg("avg_rating")).fillna(0)
result4 = pivot_stats.join(movies_df, "MovieID").select("Title", "0-18", "18-35", "35-50", "50+").orderBy("Title")
print("\nBài 4 - Kết quả:")
for row in result4.collect():
    print(f"{row.Title}: [0-18: {row['0-18']:.2f}, 18-35: {row['18-35']:.2f}, 35-50: {row['35-50']:.2f}, 50+: {row['50+']:.2f}]")


Bài 4 - Kết quả:


American Beauty (1999): [0-18: 0.00, 18-35: 4.50, 35-50: 3.00, 50+: 0.00]
Avatar (2009): [0-18: 0.00, 18-35: 4.50, 35-50: 5.00, 50+: 0.00]
Back to the Future (1985): [0-18: 0.00, 18-35: 4.00, 35-50: 0.00, 50+: 0.00]
Birdman (2014): [0-18: 0.00, 18-35: 3.00, 35-50: 0.00, 50+: 0.00]
Braveheart (1995): [0-18: 0.00, 18-35: 4.00, 35-50: 4.50, 50+: 0.00]
Coco (2017): [0-18: 0.00, 18-35: 4.25, 35-50: 0.00, 50+: 0.00]
Dunkirk (2017): [0-18: 0.00, 18-35: 3.50, 35-50: 0.00, 50+: 0.00]
Fight Club (1999): [0-18: 0.00, 18-35: 4.25, 35-50: 0.00, 50+: 0.00]
Forrest Gump (1994): [0-18: 0.00, 18-35: 5.00, 35-50: 2.50, 50+: 0.00]
Gladiator (2000): [0-18: 0.00, 18-35: 3.00, 35-50: 3.50, 50+: 0.00]
Good Will Hunting (1997): [0-18: 0.00, 18-35: 4.00, 35-50: 3.00, 50+: 0.00]
Gravity (2013): [0-18: 0.00, 18-35: 4.50, 35-50: 4.00, 50+: 0.00]
Inception (2010): [0-18: 0.00, 18-35: 3.75, 35-50: 0.00, 50+: 0.00]
Interstellar (2014): [0-18: 0.00, 18-35: 3.50, 35-50: 3.50, 50+: 0.00]
Jurassic Park (1993): [0-18: 0.